# EV Charging Stations - Bronze Layer Ingestion

**Source:** `/Volumes/mobility_ai/bronze/data_upload/ev_20251216 (1).csv`  
**Target:** `mobility_ai.bronze.ev_charging_stations_raw`  
**Mode:** Overwrite

In [0]:
# Configuration
from pyspark.sql.functions import current_timestamp, lit

# Source and Target Configuration
SOURCE_FILE = "/Volumes/mobility_ai/bronze/data_upload/ev_20251216 (1).csv"
FULL_TABLE_NAME = "mobility_ai.bronze.ev_charging_stations_raw"
SOURCE_SYSTEM = "CSV_Upload"

In [0]:
def write_dual_tables(df, catalog, schema, table_name, mode="overwrite"):
    """
    Write dataframe to both snapshot and history tables.
    Enables parallel table migration with zero downtime.
    """
    from pyspark.sql import functions as F
    
    snapshot_table = f"{catalog}.{schema}.{table_name}"
    history_table = f"{catalog}.{schema}.{table_name}_history"
    
    print(f"\n📝 Writing to dual tables: {table_name}")
    print("─" * 60)
    
    # 1. Write to snapshot table
    print(f"   1️⃣ Snapshot: {snapshot_table}")
    df.write \
        .format("delta") \
        .mode(mode) \
        .option("overwriteSchema", "true") \
        .saveAsTable(snapshot_table)
    
    snapshot_count = spark.table(snapshot_table).count()
    print(f"      ✅ {snapshot_count:,} records")
    
    # 2. Append to history table with metadata
    print(f"   2️⃣ History: {history_table}")
    
    history_df = df \
        .withColumn("ingestion_date", F.current_date()) \
        .withColumn("ingestion_timestamp_history", F.current_timestamp()) \
        .withColumn("pipeline_run_id", F.lit("manual_run"))
    
    history_df.write \
        .format("delta") \
        .mode("append") \
        .partitionBy("ingestion_date") \
        .saveAsTable(history_table)
    
    history_count = spark.table(history_table).count()
    print(f"      ✅ {history_count:,} total records")
    print("─" * 60)

In [0]:
# Read CSV and add metadata
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .load(SOURCE_FILE) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system", lit(SOURCE_SYSTEM))

record_count = df.count()

In [0]:
# Write to both snapshot and history tables (dual write pattern)
write_dual_tables(
    df=df,
    catalog="mobility_ai",
    schema="bronze",
    table_name="ev_charging_stations_raw",
    mode="overwrite"
)

In [0]:
# Validation summary
df_validation = spark.table(FULL_TABLE_NAME)

print(f"Ingestion Complete: {record_count:,} records written to {FULL_TABLE_NAME}")
display(df_validation.limit(10))

In [0]:
# Notebook complete